# Continuous Probability Distributions
### Definitions, Key Results, and Interactive Visualizations

This notebook covers the theory (definitions, PDFs, CDFs, mean, variance, and other
important results) of core continuous probability distributions, paired with
interactive widgets (sliders) so you can change parameters and immediately see how
the distribution, its CDF, mean, and variance respond.

**Contents**

1. Probability Distributions and Probability Density Functions (PDFs)
2. Cumulative Distribution Functions (CDFs)
3. Mean and Variance of a Continuous Random Variable
4. Continuous Uniform Distribution
5. Normal Distribution
6. Normal Approximation to the Binomial and Poisson Distributions
7. Exponential Distribution
8. Erlang and Gamma Distributions
9. Weibull Distribution
10. Lognormal Distribution
11. Beta Distribution


## Setup

Run this cell first. It imports everything needed and defines small helper
functions used throughout the notebook for plotting a PDF/PMF and CDF side by
side, together with the mean $\pm$ standard deviation marked on the plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gamma as gamma_fn, comb
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, fixed
from IPython.display import display, Math

plt.rcParams['figure.figsize'] = (11, 4.2)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def summary_box(ax, lines):
    """Print a small text box with numeric results (mean, var, etc.) on a plot."""
    text = "\n".join(lines)
    ax.text(0.98, 0.97, text, transform=ax.transAxes, fontsize=9,
             va='top', ha='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

def plot_continuous(x, pdf_vals, cdf_vals, mean, std, title, xlabel='x',
                     extra_lines=None, mode=None):
    """Standard two-panel plot: PDF (with mean +/- 1 std shaded) and CDF."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    ax1, ax2 = axes

    ax1.plot(x, pdf_vals, color='tab:blue', lw=2)
    ax1.fill_between(x, pdf_vals, 0, alpha=0.08, color='tab:blue')
    if std is not None and np.isfinite(std):
        mask = (x >= mean - std) & (x <= mean + std)
        ax1.fill_between(x[mask], pdf_vals[mask], 0, alpha=0.25, color='tab:orange',
                          label=r'$\mu \pm \sigma$')
    ax1.axvline(mean, color='k', ls='--', lw=1.2, label=f'mean = {mean:.3f}')
    if mode is not None and np.isfinite(mode):
        ax1.axvline(mode, color='tab:green', ls=':', lw=1.2, label=f'mode = {mode:.3f}')
    ax1.set_title(f'{title} — PDF')
    ax1.set_xlabel(xlabel); ax1.set_ylabel('f(x)')
    ax1.legend(loc='upper left', fontsize=8)

    lines = [f'mean  = {mean:.4f}']
    if std is not None and np.isfinite(std):
        lines.append(f'std   = {std:.4f}')
        lines.append(f'var   = {std**2:.4f}')
    if extra_lines:
        lines += extra_lines
    summary_box(ax1, lines)

    ax2.plot(x, cdf_vals, color='tab:red', lw=2)
    ax2.axvline(mean, color='k', ls='--', lw=1.2)
    ax2.set_ylim(-0.02, 1.02)
    ax2.set_title(f'{title} — CDF')
    ax2.set_xlabel(xlabel); ax2.set_ylabel('F(x)')

    plt.tight_layout()
    plt.show()

def plot_discrete_normal_approx(k, pmf_vals, mean, std, title, cont_correction=True):
    """Bar plot of a discrete PMF with the normal approximation overlaid."""
    fig, ax = plt.subplots(figsize=(9, 4.3))
    ax.bar(k, pmf_vals, width=0.9, alpha=0.5, color='tab:blue', label='exact PMF')
    xs = np.linspace(k.min()-1, k.max()+1, 400)
    ax.plot(xs, stats.norm.pdf(xs, mean, std), color='tab:red', lw=2,
            label='Normal approximation')
    ax.axvline(mean, color='k', ls='--', lw=1)
    ax.set_title(title)
    ax.set_xlabel('k'); ax.set_ylabel('probability')
    ax.legend(fontsize=9)
    note = "with continuity correction" if cont_correction else "no continuity correction"
    summary_box(ax, [f'mean = {mean:.3f}', f'std  = {std:.3f}', note])
    plt.tight_layout()
    plt.show()

print("Setup complete.")


## 1. Probability Distributions and Probability Density Functions

For a **continuous random variable** $X$, probabilities are described by a
**probability density function (PDF)** $f(x)$ rather than by point probabilities
(indeed $P(X=x)=0$ for every $x$).

**Definition.** A function $f(x)$ is a probability density function for the
continuous random variable $X$, defined on the set of real numbers, if

$$
1.\quad f(x) \ge 0 \quad \text{for all } x
$$
$$
2.\quad \int_{-\infty}^{\infty} f(x)\,dx = 1
$$
$$
3.\quad P(a < X < b) = \int_a^b f(x)\,dx
$$

Because $P(X=a)=0$ for continuous $X$, it makes no difference whether endpoints
are included:
$$
P(a \le X \le b) = P(a < X \le b) = P(a \le X < b) = P(a < X < b).
$$

Geometrically, $P(a<X<b)$ is the **area under the curve** $f(x)$ between $x=a$
and $x=b$.


In [ ]:

def demo_pdf_area(a=-1.0, b=1.0):
    x = np.linspace(-4, 4, 600)
    f = stats.norm.pdf(x, 0, 1)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(x, f, color='tab:blue', lw=2, label='f(x)')
    mask = (x >= a) & (x <= b)
    ax.fill_between(x[mask], f[mask], 0, color='tab:orange', alpha=0.5,
                     label=fr'$P({a:.2f}<X<{b:.2f})$')
    prob = stats.norm.cdf(b) - stats.norm.cdf(a)
    ax.set_title(f'Area under f(x) between a and b  =  P(a<X<b) = {prob:.4f}')
    ax.legend()
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    plt.tight_layout(); plt.show()

interact(demo_pdf_area,
         a=FloatSlider(value=-1, min=-4, max=4, step=0.1, description='a'),
         b=FloatSlider(value=1, min=-4, max=4, step=0.1, description='b'));


## 2. Cumulative Distribution Functions (CDFs)

**Definition.** The cumulative distribution function of a continuous random
variable $X$ is

$$
F(x) = P(X \le x) = \int_{-\infty}^{x} f(t)\,dt , \qquad -\infty < x < \infty .
$$

**Key properties**

$$
1.\quad F(-\infty) = 0, \qquad F(\infty) = 1
$$
$$
2.\quad 0 \le F(x) \le 1
$$
$$
3.\quad \text{If } x_1 \le x_2 \text{ then } F(x_1) \le F(x_2) \quad (F \text{ is non-decreasing})
$$

Given $F(x)$, the density is recovered by differentiation wherever the
derivative exists:
$$
f(x) = \frac{dF(x)}{dx}.
$$

Probabilities of intervals are differences of the CDF:
$$
P(a < X < b) = F(b) - F(a).
$$


In [ ]:

def demo_cdf(a=-1.0, b=1.0):
    x = np.linspace(-4, 4, 600)
    F = stats.norm.cdf(x)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(x, F, color='tab:red', lw=2, label='F(x)')
    Fa, Fb = stats.norm.cdf(a), stats.norm.cdf(b)
    ax.plot([a, a], [0, Fa], 'k--', lw=1)
    ax.plot([b, b], [0, Fb], 'k--', lw=1)
    ax.plot([-4, a], [Fa, Fa], color='tab:blue', lw=1, ls=':')
    ax.plot([-4, b], [Fb, Fb], color='tab:orange', lw=1, ls=':')
    ax.set_title(f'F(b) - F(a) = {Fb:.4f} - {Fa:.4f} = {Fb-Fa:.4f} = P(a<X<b)')
    ax.set_ylim(0, 1.02)
    ax.legend()
    ax.set_xlabel('x'); ax.set_ylabel('F(x)')
    plt.tight_layout(); plt.show()

interact(demo_cdf,
         a=FloatSlider(value=-1, min=-4, max=4, step=0.1, description='a'),
         b=FloatSlider(value=1, min=-4, max=4, step=0.1, description='b'));


## 3. Mean and Variance of a Continuous Random Variable

**Mean (expected value).**

$$
\mu = E(X) = \int_{-\infty}^{\infty} x\, f(x)\,dx .
$$

More generally, for a function $h(X)$,
$$
E[h(X)] = \int_{-\infty}^{\infty} h(x)\, f(x)\,dx .
$$

**Variance.**

$$
\sigma^2 = V(X) = E\big[(X-\mu)^2\big] = \int_{-\infty}^{\infty} (x-\mu)^2 f(x)\,dx
= E(X^2) - \mu^2 .
$$

**Standard deviation:** $\sigma = \sqrt{V(X)}$.

**Useful properties** (for constants $a,b$):

$$
E(aX+b) = aE(X)+b, \qquad V(aX+b) = a^2 V(X).
$$

The interactive demo below lets you pick any of the distributions from this
notebook and see $\mu$, $\sigma^2$ computed **numerically** by integrating
$x f(x)$ and $(x-\mu)^2 f(x)$, matching the closed-form results derived in the
later sections.


In [ ]:

from scipy.integrate import quad

def numeric_mean_var(pdf, lo, hi):
    mean, _ = quad(lambda x: x*pdf(x), lo, hi, limit=200)
    var, _ = quad(lambda x: (x-mean)**2*pdf(x), lo, hi, limit=200)
    return mean, var

def demo_numeric(dist_name='Normal(mu=0,sigma=1)'):
    if dist_name.startswith('Normal'):
        pdf = lambda x: stats.norm.pdf(x, 0, 1); lo, hi, xs = -8, 8, np.linspace(-4,4,400)
    elif dist_name.startswith('Uniform'):
        pdf = lambda x: stats.uniform.pdf(x, 2, 6); lo, hi, xs = 2, 8, np.linspace(2,8,400)
    elif dist_name.startswith('Exponential'):
        pdf = lambda x: stats.expon.pdf(x, scale=2); lo, hi, xs = 0, 40, np.linspace(0,15,400)
    mean, var = numeric_mean_var(pdf, lo, hi)
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(xs, pdf(xs), lw=2)
    ax.axvline(mean, color='k', ls='--')
    summary_box(ax, [f'numeric mean = {mean:.4f}', f'numeric var  = {var:.4f}'])
    ax.set_title(f'Numerical integration check: {dist_name}')
    plt.tight_layout(); plt.show()

interact(demo_numeric, dist_name=widgets.Dropdown(
    options=['Normal(mu=0,sigma=1)', 'Uniform(2,8)', 'Exponential(scale=2)'],
    description='Distribution'));


# Proof that Var(aX + b) = a²Var(X)

## Setup

Let $X$ be a continuous random variable with probability density function $f(x)$, and let $a, b \in \mathbb{R}$ be constants. Define a new random variable:

$$Y = aX + b$$

We want to prove:

$$\text{Var}(aX + b) = a^2 \text{Var}(X)$$

## Step 1: Recall the definition of variance

For any random variable $Z$:

$$\text{Var}(Z) = E\left[(Z - E[Z])^2\right]$$

## Step 2: Find E[Y]

Using linearity of expectation:

$$E[Y] = E[aX + b] = aE[X] + b$$

This follows from the definition of expectation for a continuous random variable:

$$E[aX+b] = \int_{-\infty}^{\infty} (ax+b)f(x)\,dx = a\int_{-\infty}^{\infty} xf(x)\,dx + b\int_{-\infty}^{\infty} f(x)\,dx = aE[X] + b$$

since $\int_{-\infty}^{\infty} f(x)\,dx = 1$.

## Step 3: Apply the definition of variance to Y

$$\text{Var}(Y) = E\left[(Y - E[Y])^2\right]$$

Substitute $Y = aX + b$ and $E[Y] = aE[X] + b$:

$$\text{Var}(aX+b) = E\left[\big((aX+b) - (aE[X]+b)\big)^2\right]$$

## Step 4: Simplify the expression inside the expectation

$$(aX + b) - (aE[X] + b) = aX - aE[X] = a\big(X - E[X]\big)$$

So:

$$\text{Var}(aX+b) = E\left[\big(a(X - E[X])\big)^2\right]$$

## Step 5: Expand the square

$$\text{Var}(aX+b) = E\left[a^2 (X - E[X])^2\right]$$

## Step 6: Pull the constant a² out of the expectation

Since $a^2$ is a constant, and expectation is linear:

$$E\left[a^2(X-E[X])^2\right] = a^2 E\left[(X - E[X])^2\right]$$

Explicitly, using the integral definition:

$$E\left[a^2(X-E[X])^2\right] = \int_{-\infty}^{\infty} a^2(x - E[X])^2 f(x)\,dx = a^2\int_{-\infty}^{\infty} (x-E[X])^2 f(x)\,dx$$

## Step 7: Recognize the remaining expectation as Var(X)

By definition:

$$E\left[(X - E[X])^2\right] = \text{Var}(X)$$

Therefore:

$$\text{Var}(aX+b) = a^2 \text{Var}(X)$$

$\blacksquare$

## Intuition

- **Shifting by $b$ has no effect on variance** — adding a constant translates the entire distribution but doesn't change how spread out it is around its (also shifted) mean.
- **Scaling by $a$ scales the variance by $a^2$** — variance is a measure of *squared* deviation from the mean, so scaling distances by $a$ scales the squared distances by $a^2$. This is also why standard deviation (the square root of variance) scales by $|a|$, not $a^2$.

## 4. Continuous Uniform Distribution

**Definition.** $X \sim \text{Uniform}(a,b)$ has density

$$
f(x) = \frac{1}{b-a}, \qquad a \le x \le b,
$$

and $0$ otherwise.

**CDF**

$$
F(x) =
\begin{cases}
0, & x < a \\[4pt]
\dfrac{x-a}{b-a}, & a \le x < b \\[6pt]
1, & x \ge b
\end{cases}
$$

**Mean and variance**

$$
\mu = E(X) = \frac{a+b}{2}, \qquad \sigma^2 = V(X) = \frac{(b-a)^2}{12}.
$$

**Other results**
- Every sub-interval of the same length within $[a,b]$ has the same probability
  (constant density) — this is the defining "equally likely" property.
- Moment generating function: $M_X(t) = \dfrac{e^{tb}-e^{ta}}{t(b-a)}$ for $t\neq 0$.


In [ ]:

def uniform_demo(a=0.0, b=1.0):
    x = np.linspace(a-0.3*(b-a+1e-9), b+0.3*(b-a+1e-9), 500)
    pdf_vals = stats.uniform.pdf(x, loc=a, scale=b-a)
    cdf_vals = stats.uniform.cdf(x, loc=a, scale=b-a)
    mean = (a+b)/2
    var = (b-a)**2/12
    plot_continuous(x, pdf_vals, cdf_vals, mean, np.sqrt(var),
                     title=f'Uniform(a={a:.2f}, b={b:.2f})',
                     extra_lines=[f'a={a:.2f}, b={b:.2f}'])

interact(uniform_demo,
         a=FloatSlider(value=0, min=-5, max=5, step=0.1, description='a'),
         b=FloatSlider(value=4, min=-5, max=10, step=0.1, description='b'));


## 5. Normal Distribution

**Definition.** $X \sim N(\mu, \sigma^2)$ has density

$$
f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left[-\frac{(x-\mu)^2}{2\sigma^2}\right],
\qquad -\infty < x < \infty .
$$

**Mean and variance**
$$
E(X) = \mu, \qquad V(X) = \sigma^2 .
$$

**Standard normal.** If $Z = \dfrac{X-\mu}{\sigma}$ then $Z \sim N(0,1)$ with

$$
\phi(z) = \frac{1}{\sqrt{2\pi}} e^{-z^2/2}, \qquad
\Phi(z) = P(Z\le z) = \int_{-\infty}^{z} \phi(t)\,dt .
$$

so that
$$
P(X \le x) = \Phi\!\left(\frac{x-\mu}{\sigma}\right).
$$

**Empirical (68–95–99.7) rule**
$$
P(\mu-\sigma < X < \mu+\sigma) \approx 0.6827, \quad
P(\mu-2\sigma < X < \mu+2\sigma) \approx 0.9545, \quad
P(\mu-3\sigma < X < \mu+3\sigma) \approx 0.9973.
$$

**Other results**
- The normal curve is symmetric about $\mu$ and bell-shaped; its two inflection
  points occur at $x=\mu\pm\sigma$.
- Sums of independent normals are normal: if $X_i \sim N(\mu_i,\sigma_i^2)$
  independent, then $\sum X_i \sim N\!\left(\sum \mu_i, \sum \sigma_i^2\right)$.


### Proving ∫ f(x) dx = 1 for the Normal Distribution

**Goal:** Show that for the normal PDF

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

we have $\int_{-\infty}^{\infty} f(x)\,dx = 1$.

Substitute $z = \frac{x-\mu}{\sigma}$, so $dx = \sigma\,dz$:

$$\int_{-\infty}^{\infty} f(x)\,dx = \frac{1}{\sigma\sqrt{2\pi}}\int_{-\infty}^{\infty} e^{-z^2/2}\,\sigma\,dz = \frac{1}{\sqrt{2\pi}}\int_{-\infty}^{\infty} e^{-z^2/2}\,dz$$

So it suffices to show the **Gaussian integral**:

$$I = \int_{-\infty}^{\infty} e^{-z^2/2}\,dz = \sqrt{2\pi}$$

The function $e^{-z^2/2}$ has no elementary antiderivative, so we can't integrate it directly. Instead, consider $I^2$ as a **double integral**:

$$I^2 = \left(\int_{-\infty}^{\infty} e^{-x^2/2}\,dx\right)\left(\int_{-\infty}^{\infty} e^{-y^2/2}\,dy\right) = \int_{-\infty}^{\infty}\int_{-\infty}^{\infty} e^{-(x^2+y^2)/2}\,dx\,dy$$

This is now an integral over the entire $xy$-plane — perfect for polar coordinates.

Let $x = r\cos\theta$, $y = r\sin\theta$, so $x^2+y^2 = r^2$ and $dx\,dy = r\,dr\,d\theta$:

$$I^2 = \int_0^{2\pi}\int_0^{\infty} e^{-r^2/2}\, r\,dr\,d\theta$$

Now the $r$-integral **does** have an elementary antiderivative, since the extra factor of $r$ makes it a simple substitution. Let $u = r^2/2$, $du = r\,dr$:

$$\int_0^{\infty} e^{-r^2/2}\,r\,dr = \int_0^{\infty} e^{-u}\,du = 1$$

So:

$$I^2 = \int_0^{2\pi} 1 \, d\theta = 2\pi$$

$$I = \sqrt{2\pi}$$

Therefore:

$$\int_{-\infty}^{\infty} f(x)\,dx = \frac{1}{\sqrt{2\pi}} \cdot \sqrt{2\pi} = 1 \qquad \blacksquare$$

**Key insight:** this is the classic workaround to the fact that $e^{-x^2}$ has no elementary antiderivative in 1D — squaring the integral and switching to polar coordinates introduces a factor of $r$ that *does* integrate elementarily, sidestepping the issue entirely.

In [ ]:

def normal_demo(mu=0.0, sigma=1.0):
    x = np.linspace(mu-4.5*sigma, mu+4.5*sigma, 500)
    pdf_vals = stats.norm.pdf(x, mu, sigma)
    cdf_vals = stats.norm.cdf(x, mu, sigma)
    plot_continuous(x, pdf_vals, cdf_vals, mu, sigma,
                     title=f'Normal(mu={mu:.2f}, sigma={sigma:.2f})',
                     extra_lines=[
                         f'P(mu-sigma<X<mu+sigma)={stats.norm.cdf(1)-stats.norm.cdf(-1):.4f}',
                         f'P(mu-2s<X<mu+2s)={stats.norm.cdf(2)-stats.norm.cdf(-2):.4f}'
                     ])

interact(normal_demo,
         mu=FloatSlider(value=0, min=-10, max=10, step=0.5, description='mu'),
         sigma=FloatSlider(value=1, min=0.1, max=5, step=0.1, description='sigma'));


## 6. Normal Approximation to the Binomial and Poisson Distributions

### Binomial $\to$ Normal

If $X \sim \text{Binomial}(n,p)$, then $E(X)=np$, $V(X)=np(1-p)$, and for large
$n$ (rule of thumb: $np>5$ and $n(1-p)>5$),

$$
Z = \frac{X-np}{\sqrt{np(1-p)}} \ \xrightarrow{\ d\ }\ N(0,1).
$$

With the **continuity correction**,
$$
P(X \le x) \approx \Phi\!\left(\frac{x+0.5-np}{\sqrt{np(1-p)}}\right).
$$

### Poisson $\to$ Normal

If $X \sim \text{Poisson}(\lambda)$, then $E(X)=\lambda$, $V(X)=\lambda$, and for
large $\lambda$ (rule of thumb: $\lambda>5$),
$$
Z = \frac{X-\lambda}{\sqrt{\lambda}} \ \xrightarrow{\ d\ }\ N(0,1),
$$
again with a $\pm 0.5$ continuity correction when approximating a discrete sum.


In [ ]:

def binom_normal_demo(n=40, p=0.3):
    k = np.arange(0, n+1)
    pmf_vals = stats.binom.pmf(k, n, p)
    mean, std = n*p, np.sqrt(n*p*(1-p))
    lo = max(0, int(mean-4*std)); hi = min(n, int(mean+4*std)+1)
    plot_discrete_normal_approx(k[lo:hi], pmf_vals[lo:hi], mean, std,
                                 f'Binomial(n={n}, p={p:.2f}) vs Normal approx  (np={mean:.1f}, np(1-p)={std**2:.2f})')

interact(binom_normal_demo,
         n=IntSlider(value=40, min=5, max=200, step=1, description='n'),
         p=FloatSlider(value=0.3, min=0.01, max=0.99, step=0.01, description='p'));


In [ ]:

def poisson_normal_demo(lam=15):
    k = np.arange(0, max(30, int(lam*3)))
    pmf_vals = stats.poisson.pmf(k, lam)
    mean, std = lam, np.sqrt(lam)
    lo = max(0, int(mean-4*std)); hi = min(len(k), int(mean+4*std)+1)
    plot_discrete_normal_approx(k[lo:hi], pmf_vals[lo:hi], mean, std,
                                 f'Poisson(lambda={lam}) vs Normal approx  (mean=var={lam})')

interact(poisson_normal_demo,
         lam=FloatSlider(value=15, min=1, max=100, step=1, description='lambda'));


## 7. Exponential Distribution

**Definition.** $X \sim \text{Exponential}(\lambda)$, $\lambda>0$, has density

$$
f(x) = \lambda e^{-\lambda x}, \qquad x \ge 0 .
$$

**CDF**
$$
F(x) = 1 - e^{-\lambda x}, \qquad x \ge 0 .
$$

**Mean and variance**
$$
\mu = E(X) = \frac{1}{\lambda}, \qquad \sigma^2 = V(X) = \frac{1}{\lambda^2}.
$$

**Other results**
- **Memoryless property:** $P(X > t+s \mid X>t) = P(X>s)$ for all $s,t \ge 0$.
- The exponential distribution models the waiting time between events of a
  **Poisson process** with rate $\lambda$: if events occur at rate $\lambda$
  per unit time (Poisson counts), the time between consecutive events is
  $\text{Exponential}(\lambda)$.
- Reliability/hazard function is constant: $h(x) = f(x)/[1-F(x)] = \lambda$.


In [ ]:

def exponential_demo(lam=1.0):
    mean = 1/lam
    x = np.linspace(0, mean*6+0.5, 500)
    pdf_vals = stats.expon.pdf(x, scale=1/lam)
    cdf_vals = stats.expon.cdf(x, scale=1/lam)
    plot_continuous(x, pdf_vals, cdf_vals, mean, mean,
                     title=f'Exponential(lambda={lam:.2f})', xlabel='x (>=0)',
                     extra_lines=[f'1/lambda = {mean:.4f}', 'memoryless: P(X>t+s|X>t)=P(X>s)'])

interact(exponential_demo,
         lam=FloatSlider(value=1.0, min=0.05, max=5, step=0.05, description='lambda'));


## 8. Erlang and Gamma Distributions

**Gamma function.**
$$
\Gamma(r) = \int_0^\infty x^{r-1} e^{-x}\,dx, \qquad r>0, \qquad
\Gamma(r) = (r-1)\Gamma(r-1), \qquad \Gamma(n) = (n-1)! \ (n \in \mathbb{Z}^+).
$$

**Definition (Gamma distribution).** $X \sim \text{Gamma}(r,\lambda)$, with
shape $r>0$ and rate $\lambda>0$, has density

$$
f(x) = \frac{\lambda^{r} x^{r-1} e^{-\lambda x}}{\Gamma(r)}, \qquad x \ge 0 .
$$

**Mean and variance**
$$
\mu = E(X) = \frac{r}{\lambda}, \qquad \sigma^2 = V(X) = \frac{r}{\lambda^2}.
$$

**Erlang distribution.** The special case where $r=k$ is a **positive
integer** is the Erlang distribution; it is the distribution of the sum of
$k$ independent $\text{Exponential}(\lambda)$ random variables — i.e. the
waiting time until the $k$-th event of a Poisson process with rate
$\lambda$:
$$
X = X_1+X_2+\dots+X_k, \qquad X_i \overset{iid}{\sim} \text{Exponential}(\lambda).
$$
For integer $r=k$, $\Gamma(k)=(k-1)!$, and the CDF has a closed (Poisson-sum)
form:
$$
F(x) = 1 - \sum_{j=0}^{k-1} \frac{e^{-\lambda x}(\lambda x)^j}{j!}, \qquad x\ge0.
$$

**Other results**
- Gamma($r=1,\lambda$) $=$ Exponential($\lambda$).
- Sum of independent Gammas with the same rate: if $X_i\sim\text{Gamma}(r_i,\lambda)$
  independent, $\sum X_i \sim \text{Gamma}\!\left(\sum r_i, \lambda\right)$.


In [ ]:

def gamma_demo(r=3.0, lam=1.0):
    mean, var = r/lam, r/lam**2
    std = np.sqrt(var)
    x = np.linspace(1e-6, mean+6*std+1, 500)
    pdf_vals = stats.gamma.pdf(x, a=r, scale=1/lam)
    cdf_vals = stats.gamma.cdf(x, a=r, scale=1/lam)
    mode = (r-1)/lam if r>=1 else np.nan
    is_erlang = abs(r-round(r))<1e-9 and r>=1
    plot_continuous(x, pdf_vals, cdf_vals, mean, std,
                     title=f'Gamma(r={r:.2f}, lambda={lam:.2f})' + (' [Erlang: integer r]' if is_erlang else ''),
                     xlabel='x (>=0)', mode=mode,
                     extra_lines=[f'r/lambda={mean:.3f}', f'r/lambda^2={var:.3f}'])

interact(gamma_demo,
         r=FloatSlider(value=3.0, min=0.2, max=15, step=0.1, description='r (shape)'),
         lam=FloatSlider(value=1.0, min=0.1, max=5, step=0.1, description='lambda (rate)'));


## 9. Weibull Distribution

**Definition.** $X \sim \text{Weibull}(\beta,\delta)$, shape $\beta>0$, scale
$\delta>0$, has density

$$
f(x) = \frac{\beta}{\delta}\left(\frac{x}{\delta}\right)^{\beta-1}
\exp\!\left[-\left(\frac{x}{\delta}\right)^{\beta}\right], \qquad x \ge 0.
$$

**CDF**
$$
F(x) = 1 - \exp\!\left[-\left(\frac{x}{\delta}\right)^{\beta}\right], \qquad x\ge 0.
$$

**Mean and variance**
$$
\mu = E(X) = \delta\, \Gamma\!\left(1+\frac{1}{\beta}\right), \qquad
\sigma^2 = V(X) = \delta^2\left[\Gamma\!\left(1+\frac{2}{\beta}\right)
- \Gamma\!\left(1+\frac{1}{\beta}\right)^2\right].
$$

**Other results**
- $\beta=1$ reduces to the Exponential($\lambda=1/\delta$) distribution.
- The hazard function $h(x)=\dfrac{\beta}{\delta}\left(\dfrac{x}{\delta}\right)^{\beta-1}$
  is increasing for $\beta>1$, decreasing for $\beta<1$, and constant
  ($=\lambda$) for $\beta=1$ — this is why Weibull is widely used in
  reliability/failure-time modeling (wear-out vs. infant-mortality failures).


In [ ]:

def weibull_demo(beta=2.0, delta=1.0):
    mean = delta*gamma_fn(1+1/beta)
    var = delta**2*(gamma_fn(1+2/beta) - gamma_fn(1+1/beta)**2)
    std = np.sqrt(var)
    x = np.linspace(1e-6, mean+6*std+1, 500)
    pdf_vals = stats.weibull_min.pdf(x, c=beta, scale=delta)
    cdf_vals = stats.weibull_min.cdf(x, c=beta, scale=delta)
    mode = delta*((beta-1)/beta)**(1/beta) if beta>1 else 0.0
    plot_continuous(x, pdf_vals, cdf_vals, mean, std,
                     title=f'Weibull(beta={beta:.2f}, delta={delta:.2f})',
                     xlabel='x (>=0)', mode=mode,
                     extra_lines=[f'hazard: {"increasing" if beta>1 else ("decreasing" if beta<1 else "constant")}'])

interact(weibull_demo,
         beta=FloatSlider(value=2.0, min=0.2, max=8, step=0.1, description='beta (shape)'),
         delta=FloatSlider(value=1.0, min=0.1, max=6, step=0.1, description='delta (scale)'));


## 10. Lognormal Distribution

**Definition.** $X$ is lognormal if $Y=\ln(X) \sim N(\theta,\omega^2)$. Its
density is

$$
f(x) = \frac{1}{x\,\omega\sqrt{2\pi}}
\exp\!\left[-\frac{(\ln x-\theta)^2}{2\omega^2}\right], \qquad x > 0.
$$

**CDF** (via the standard normal CDF $\Phi$)
$$
F(x) = \Phi\!\left(\frac{\ln x-\theta}{\omega}\right), \qquad x>0.
$$

**Mean and variance**
$$
\mu = E(X) = e^{\theta+\omega^2/2}, \qquad
\sigma^2 = V(X) = e^{2\theta+\omega^2}\left(e^{\omega^2}-1\right).
$$

**Other results**
- $\theta$ and $\omega^2$ are the mean and variance of $\ln X$, **not** of $X$
  itself.
- Median of $X$ is $e^{\theta}$; the distribution is right-skewed.
- Products of independent lognormals are lognormal — useful for modeling
  quantities built from multiplicative effects (e.g. many manufacturing and
  financial variables).


In [ ]:

def lognormal_demo(theta=0.0, omega=0.5):
    mean = np.exp(theta+omega**2/2)
    var = np.exp(2*theta+omega**2)*(np.exp(omega**2)-1)
    std = np.sqrt(var)
    median = np.exp(theta)
    x = np.linspace(1e-6, mean+6*std+1, 500)
    pdf_vals = stats.lognorm.pdf(x, s=omega, scale=np.exp(theta))
    cdf_vals = stats.lognorm.cdf(x, s=omega, scale=np.exp(theta))
    plot_continuous(x, pdf_vals, cdf_vals, mean, std,
                     title=f'Lognormal(theta={theta:.2f}, omega={omega:.2f})',
                     xlabel='x (>0)',
                     extra_lines=[f'median = e^theta = {median:.3f}'])

interact(lognormal_demo,
         theta=FloatSlider(value=0.0, min=-2, max=2, step=0.1, description='theta'),
         omega=FloatSlider(value=0.5, min=0.05, max=2, step=0.05, description='omega'));


## 11. Beta Distribution

**Definition.** $X \sim \text{Beta}(\alpha,\beta)$ on $[0,1]$ has density

$$
f(x) = \frac{\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\beta)}\,
x^{\alpha-1}(1-x)^{\beta-1}, \qquad 0 \le x \le 1, \ \ \alpha,\beta>0.
$$

**Mean and variance**
$$
\mu = E(X) = \frac{\alpha}{\alpha+\beta}, \qquad
\sigma^2 = V(X) = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}.
$$

**Generalized Beta on $[a,b]$.** With $X' = a+(b-a)X$,
$$
f(x') = \frac{\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\beta)}
\frac{(x'-a)^{\alpha-1}(b-x')^{\beta-1}}{(b-a)^{\alpha+\beta-1}}, \qquad a\le x'\le b,
$$
$$
E(X') = a+(b-a)\frac{\alpha}{\alpha+\beta}, \qquad
V(X') = (b-a)^2\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)} .
$$

**Other results**
- $\alpha=\beta=1$ reduces to Uniform$(0,1)$.
- $\alpha=\beta$ gives a distribution symmetric about $0.5$.
- Mode (for $\alpha,\beta>1$): $\dfrac{\alpha-1}{\alpha+\beta-2}$.
- The Beta distribution is the conjugate prior for a Binomial proportion $p$
  in Bayesian inference, which is why $\alpha,\beta$ are often interpreted as
  "prior successes/failures."


In [ ]:

def beta_demo(alpha=2.0, beta_param=2.0):
    mean = alpha/(alpha+beta_param)
    var = (alpha*beta_param)/((alpha+beta_param)**2*(alpha+beta_param+1))
    std = np.sqrt(var)
    x = np.linspace(1e-6, 1-1e-6, 500)
    pdf_vals = stats.beta.pdf(x, alpha, beta_param)
    cdf_vals = stats.beta.cdf(x, alpha, beta_param)
    mode = (alpha-1)/(alpha+beta_param-2) if alpha>1 and beta_param>1 else np.nan
    plot_continuous(x, pdf_vals, cdf_vals, mean, std,
                     title=f'Beta(alpha={alpha:.2f}, beta={beta_param:.2f})',
                     xlabel='x in [0,1]', mode=mode)

interact(beta_demo,
         alpha=FloatSlider(value=2.0, min=0.2, max=10, step=0.1, description='alpha'),
         beta_param=FloatSlider(value=2.0, min=0.2, max=10, step=0.1, description='beta'));


## Summary Table

| Distribution | PDF $f(x)$ | Mean | Variance |
|---|---|---|---|
| Uniform $(a,b)$ | $\frac{1}{b-a}$ | $\frac{a+b}{2}$ | $\frac{(b-a)^2}{12}$ |
| Normal $(\mu,\sigma^2)$ | $\frac{1}{\sigma\sqrt{2\pi}}e^{-(x-\mu)^2/2\sigma^2}$ | $\mu$ | $\sigma^2$ |
| Exponential$(\lambda)$ | $\lambda e^{-\lambda x}$ | $1/\lambda$ | $1/\lambda^2$ |
| Gamma/Erlang$(r,\lambda)$ | $\frac{\lambda^r x^{r-1}e^{-\lambda x}}{\Gamma(r)}$ | $r/\lambda$ | $r/\lambda^2$ |
| Weibull$(\beta,\delta)$ | $\frac{\beta}{\delta}(x/\delta)^{\beta-1}e^{-(x/\delta)^\beta}$ | $\delta\Gamma(1+1/\beta)$ | $\delta^2[\Gamma(1+2/\beta)-\Gamma(1+1/\beta)^2]$ |
| Lognormal$(\theta,\omega^2)$ | $\frac{1}{x\omega\sqrt{2\pi}}e^{-(\ln x-\theta)^2/2\omega^2}$ | $e^{\theta+\omega^2/2}$ | $e^{2\theta+\omega^2}(e^{\omega^2}-1)$ |
| Beta$(\alpha,\beta)$ | $\frac{\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\beta)}x^{\alpha-1}(1-x)^{\beta-1}$ | $\frac{\alpha}{\alpha+\beta}$ | $\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$ |

**Tip:** Every interactive cell above uses `ipywidgets.interact`, so moving a
slider re-draws the PDF/PMF and CDF panels instantly along with the numeric
mean/variance (and other quantities such as mode or hazard behavior) computed
directly from the closed-form formulas.
